In [1]:
# for reading env file
from dotenv import load_dotenv
import os

# for loading db
import pymysql

# for working part
import numpy as np
import pandas as pd


# for config file 
import json

# for parquet files
import fastparquet

In [2]:
## reading my sql database through env

load_dotenv()



conn = pymysql.connect(
    host=os.getenv("host"),
    port=int(os.getenv("port")),
    user=os.getenv("user"),
    password=os.getenv("password"),
    database=os.getenv("database"),
    ssl={"ssl_mode": "REQUIRED"}
)

cursor = conn.cursor()



print("the db is connected database name is ",os.getenv("database"))

the db is connected database name is  pharma_db


In [3]:
# reading the config file for selected tables and their selecetd columns


with open("../config/selected_tables.json") as f:
    tables_config = json.load(f)

with open("../config/selected_columns.json") as f:
    columns_config = json.load(f)

In [4]:
# function to load selected tables with their selected columns


def load_tables(selected_tables, columns_config):

    data = {}

    for table in selected_tables:

        cols = ", ".join(columns_config[table])

        query = f"""
        SELECT {cols}
        FROM {table}
        """

        data[table] = pd.read_sql(
            query,
            conn
        )

    return data

In [5]:
# calling the above load tables function

selected_tables = tables_config["selected_tables"]

data = load_tables(
    selected_tables,
    columns_config
)

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_24768\2457366214.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(


In [6]:
# getting the table names

data.keys()

dict_keys(['calls', 'contacts', 'leads', 'orders', 'pii'])

In [7]:
# taking out the tables from the dictionary and creating a dataframe

leads = data["leads"]
contacts = data["contacts"]
pii=data['pii']
calls = data["calls"]
orders = data['orders']

In [8]:
# leads merged with contacts and pii for their table details important for the project, connected through pii_id

lead_conversion_datset = leads.merge(contacts,how="left",on = "pii_id").merge(pii,how='left',on='pii_id')



In [9]:
# creating a aggregation df of calls as calls table can be transformed into this easily and is more important this way then the raw table


calls_data = calls.groupby('pii_id').agg(total_duration = ("duration","sum"),call_count = ('call_id',"count"),
                            last_call_date = ("call_date","max"),first_call_date=("call_date","min"),distinct_call_days = ('call_date','nunique'),
                            connected_call_count = ('outcome',lambda x : (x=="connected").sum()),
                            missed_call_count = ('outcome',lambda x : (x=="missed").sum()),
                            inbound_call_count = ('call_type',lambda x: (x=='inbound').sum()),
                            outbound_call_count = ('call_type',lambda x: (x=='outbound').sum()))

In [10]:
# merging the lead conversion datset which we created by merging above with calls aggregation data for more features

lead_conversion_datset = lead_conversion_datset.merge(calls_data,how = "left", on = "pii_id")

In [11]:
# adding the output columns in the dataset by creating a list of converted pii_id from orders table and matching with the final merged db


converted_pii = orders['pii_id'].unique().tolist()


lead_conversion_datset['converted'] = lead_conversion_datset['pii_id'].isin(converted_pii).astype(int)


In [12]:
# date columns have none values to convert into nat for single dtype in columns we are updating all date columns


date_cols = [
    'assigned_date',
    'follow_up_date',
    'first_call_date',
    'last_call_date'
]

for col in date_cols:
    lead_conversion_datset[col] = pd.to_datetime(
        lead_conversion_datset[col],
        errors='coerce'
    )

In [13]:
# finally loading the dataset into parquet file
## parquet file stores data in binary format, takes low space, and can be read by machine fast


lead_conversion_datset.to_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\raw_data.parquet",engine="fastparquet",index=False)